# Episode 3 — Automatic Differentiation

**Instructor notebook** · run top-to-bottom before recording.

`jax.grad` for scalars; `value_and_grad` for training; `jax.checkpoint` to trade compute for memory.

| | |
|---|---|
| **Chapter** | 1.3 · Part I — Pure JAX |
| **Prereq** | Episodes 1–2 |
| **Next** | Episode 4 — `vmap`, `scan`, and vectorization |

**JAX docs:** [Automatic differentiation](https://docs.jax.dev/en/latest/automatic-differentiation.html) · [`jax.grad`](https://docs.jax.dev/en/latest/_autosummary/jax.grad.html) · [`jax.vjp`](https://docs.jax.dev/en/latest/_autosummary/jax.vjp.html) · [`jax.value_and_grad`](https://docs.jax.dev/en/latest/_autosummary/jax.value_and_grad.html) · [`jax.checkpoint`](https://docs.jax.dev/en/latest/_autosummary/jax.checkpoint.html)


In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from jax import grad, value_and_grad

## `grad` — scalar outputs only

`jax.grad(f)` returns a function that computes ∂f/∂x. The output of `f` must be a **scalar** (a 0-D array).

In [ ]:
def loss_wrt_W(W, x):
    probs = jax.nn.softmax(x)
    return jnp.sum(probs @ W)


x = jnp.array([1.0, 2.0, 0.5])
W = jnp.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])

grad_W = grad(loss_wrt_W, argnums=0)(W, x)
print("grad W shape:", grad_W.shape)
print(grad_W)

## `value_and_grad` — loss and gradients together

In [ ]:
def ce_loss(params, x, y):
    logits = x @ params
    log_probs = jax.nn.log_softmax(logits)
    return -log_probs[y]


params = jr.normal(jr.key(0), (4, 3))
x = jr.normal(jr.key(1), (4,))
y = jnp.int32(1)

loss, grads = value_and_grad(ce_loss)(params, x, y)
print("loss:", loss)
print("grads shape:", grads.shape)

## Check against finite differences

In [ ]:
def finite_diff_grad(f, x, eps=1e-4):
    g = jnp.zeros_like(x)
    for idx in jnp.ndindex(x.shape):
        xp = x.at[idx].add(eps)
        xm = x.at[idx].add(-eps)
        g = g.at[idx].set((f(xp) - f(xm)) / (2 * eps))
    return g


_, ad_grad = value_and_grad(ce_loss)(params, x, y)
fd_grad = finite_diff_grad(lambda p: ce_loss(p, x, y), params)
print("max |AD - FD|:", float(jnp.max(jnp.abs(ad_grad - fd_grad))))

## `vjp` when the output is not a scalar

`jax.grad` requires a **scalar** loss. For vector- or tuple-valued functions, use `jax.vjp` — it returns the output plus a function that maps a **cotangent** (same shape as the output) back to input gradients.

In [ ]:
def vec_out(x):
    return jnp.array([jnp.sum(x ** 2), jnp.sum(x)])


x = jnp.array([1.0, 2.0, 3.0])

try:
    grad(vec_out)(x)
except TypeError as e:
    print("grad on vector output:", e)

out, vjp_fn = jax.vjp(vec_out, x)
# Cotangent selects which output component to backprop — here weight both equally
grad_x = vjp_fn(jnp.array([1.0, 1.0]))
print("out:", out)
print("vjp grad:", grad_x)

## Gradient checkpointing

`jax.checkpoint` re-runs forward pieces during the backward pass instead of storing every activation. Same forward **value**, less memory, more compute.

In [ ]:
def deep_chain(x, depth=8):
    for _ in range(depth):
        x = jnp.tanh(x @ jnp.eye(x.shape[-1]))
    return jnp.sum(x)


x = jr.normal(jr.key(2), (32, 32))
checkpointed = jax.checkpoint(deep_chain)

loss_plain, _ = value_and_grad(deep_chain)(x)
loss_ckpt, _ = value_and_grad(checkpointed)(x)
print("same loss:", jnp.allclose(loss_plain, loss_ckpt))

> **Key insight:** Use `value_and_grad` for scalar training losses. Reach for `vjp` when the forward pass returns multiple outputs or you need a custom cotangent.

---
## Exercise

1. Verify `grad` of `softmax(x) @ W` on a tiny example.
2. Implement CE loss + `value_and_grad`; compare to finite differences.
3. Call `jax.vjp` on a vector-valued function with a cotangent.
4. Wrap a deep function with `jax.checkpoint` and confirm the forward value is unchanged.

*(Solution below.)*

In [ ]:
print('loss plain:', float(loss_plain), '| checkpointed:', float(loss_ckpt))

**Next:** Episode 4 — `vmap`, `scan`, and vectorization.